# torch.distributed.all_gather 和 torch.distributed.all_gather_object 
- 都是 PyTorch 分布式通信中用于“全收集（All-Gather）”的核心 API。它们的作用都是将集群中所有进程（如不同 GPU 上的 rank）的数据收集起来，并同步给每一个进程。

它们的核心区别:<font color='red'>在于收集的数据类型和底层实现机制：</font>
- all_gather：<font color='green'>专门用于收集 torch.Tensor（张量）<font color='red'>，底层调用高性能的通信后端（如 NCCL），效率极高。
- all_gather_object：<font color='green'>用于收集任意 Python 对象（如字典、列表、字符串、数字等）</font>，底层依赖 Python 的 pickle 序列化，灵活但性能较低。

下面为你详细拆解两者的区别与适用场景：
## 🚀 torch.distributed.all_gather（高性能张量收集）
这是分布式训练中最常用的通信原语，专为深度学习中的张量数据设计。
    - 数据类型：<font color='red'>只能收集 torch.Tensor。</font>
- 底层机制：<font color='red'>直接利用底层的分布式通信库（如 NVIDIA 的 NCCL 或 Gloo）进行内存级别的二进制数据搬运，速度极快，且支持梯度传播（在特定配置下）。</font>
- 核心限制：要求所有进程发送的张量形状（shape）和数据类型（dtype）必须完全一致（或者提前预分配好对应大小的空张量作为接收容器）。
- 收集顺序：严格按照进程的 rank 顺序（从小到大，即 rank 0, rank 1, rank 2...）将张量放入输出列表中。

###### 适用场景：
- 分布式训练中的<font color='red'>梯度同步、参数同步</font>。
- 模型评估时，<font color='red'>收集各个 GPU 上的预测结果张量（logits）、标签张量（labels）以计算全局准确率</font>。
- 任何涉及大规模数值矩阵或向量的高效通信。

In [ ]:
import torch
import torch.distributed as dist

# 假设 world_size = 2
rank = dist.get_rank()

# 1. 准备要发送的张量 (每个 rank 发送自己的数据)
local_tensor = torch.tensor([rank * 10], device='cuda') 

# 2. 预分配接收容器 (长度等于 world_size，每个元素是与 local_tensor 形状相同的空张量)
gathered_tensors = [torch.empty_like(local_tensor) for _ in range(dist.get_world_size())]

# 3. 执行 all_gather
dist.all_gather(gathered_tensors, local_tensor)

# 结果：所有 rank 的 gathered_tensors 都会变成 [tensor([0]), tensor([10])]

## 📦 torch.distributed.all_gather_object（灵活的通用对象收集）
当你要收集的数据不是规整的张量，而是复杂的 Python 原生结构时，就需要用到它。
- 数据类型：任意可以被 Python pickle 模块序列化的对象（如 dict, list, str, int, 自定义类的实例等）。
- 底层机制：<font color='red'>先将每个进程的 Python 对象通过 pickle 序列化成字节流，然后通过通信后端交换这些字节流，最后在目标进程上反序列化回 Python 对象。</font>
- 核心优势：<font color='red'>极其灵活，不受数据形状的限制，非常适合传输非数值型的元数据。</font>
- 性能短板：<font color='green'>由于涉及序列化和反序列化，且无法利用 NCCL 的底层优化，通信开销和 CPU 占用远高于 all_gather。</font>

###### 适用场景：
- <font color='red'>收集各个进程的日志信息、配置参数（config）、超参数。</font>
- 在分布式推理时，收集各个 rank 生成的文本字符串（string）或推理结果的字典，以便在 rank 0 上统一打印或保存。
- 传输无法轻易转换为张量的复杂元数据。

In [ ]:
import torch.distributed as dist

rank = dist.get_rank()

# 1. 准备要发送的任意 Python 对象 (比如一个包含 rank 信息的字典)
local_data = {"rank": rank, "status": "success", "metric": 0.95}

# 2. 预分配接收容器 (长度等于 world_size，用 None 占位即可)
gathered_results = [None for _ in range(dist.get_world_size())]

# 3. 执行 all_gather_object
dist.all_gather_object(gathered_results, local_data)

# 结果：所有 rank 的 gathered_results 都会变成：
# [{'rank': 0, 'status': 'success', 'metric': 0.95}, {'rank': 1, 'status': 'success', 'metric': 0.95}, ...]

| 维度 | `all_gather` | `all_gather_object` |
| :--- | :--- | :--- |
| 数据类型 | 仅限 `torch.Tensor` | 任意可 `pickle` 的 Python 对象 |
| 底层机制 | 底层通信库 (如 NCCL) 直接搬运内存 | `pickle` 序列化/反序列化 + 通信 |
| 性能表现 | 极高 (适合大规模数值计算) | 较低 (受限于序列化和带宽) |
| 形状限制 | 发送的张量形状必须一致 | 无限制，每个对象可以完全不同 |
| 典型场景 | 梯度/参数同步、评估指标计算 | 收集日志、配置、文本输出、元数据 |

## 💡 最佳实践建议：
- 在分布式大模型开发中，能转成 Tensor 的尽量用 all_gather，以保证极致的通信效率；只有在处理日志、配置或文本等无法向量化的“杂项数据”时，才退而求其次使用 all_gather_object。

# Pickle

- pickle 是 Python 标准库中自带的一个极其强大的对象序列化模块。你可以把它想象成 Python 对象的“冷冻休眠舱”：它能把内存中正在运行的复杂 Python 对象（比如训练好的模型、字典、列表、自定义类实例等）完整地“冷冻”成一串二进制字节流；等到需要时，又能通过“解冻”完美还原出和原来一模一样的对象。

## 🛠️ 核心用法：序列化与反序列化
pickle 的使用非常直观，主要包含四个核心函数，分为“文件操作”和“内存操作”两类：
- dump / load：直接将对象序列化并保存到文件，或从文件中读取并还原对象。
- dumps / loads：将对象序列化成内存中的 bytes 字节流，或从字节流中还原对象（常用于网络传输或缓存）。